# Lab 08 — When Information Is Lost

## Rank, null space, column space, and invertibility

In Chapter 7, we learned to solve backwards: given a matrix machine $A$ and an output $b$, find an input $x$ such that $Ax=b$.

In this lab, we ask a deeper question:

> When is solving backwards possible, unique, unstable, or impossible?

The answer depends on what information the matrix preserves and what information it destroys.

This lab is longer than a quick practice sheet. It is designed as a guided computational story. You will use Python to explore:

- collapse maps in 2D,
- column spaces and reachable targets,
- null spaces and invisible directions,
- rank and rank-nullity,
- consistent, inconsistent, and infinitely-many-solution systems,
- numerical near-information-loss,
- image compression as controlled loss,
- high-dimensional random projections.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)


## 1. A matrix that destroys one coordinate

The matrix

$$
P = \begin{bmatrix}1&0\\0&0\end{bmatrix}
$$

keeps the horizontal coordinate and erases the vertical coordinate.


In [ ]:
P = np.array([[1, 0],
              [0, 0]], dtype=float)

points = np.array([[3, 2], [3, 100], [-1, 4], [-1, -7]], dtype=float)
images = points @ P.T

print("Original points:")
print(points)
print("\nAfter applying P:")
print(images)


In [ ]:
def plot_before_after(A, pts, title="Transformation"):
    img = pts @ A.T
    plt.figure(figsize=(7,7))
    plt.axhline(0, linewidth=1)
    plt.axvline(0, linewidth=1)
    plt.scatter(pts[:,0], pts[:,1], label="original inputs")
    plt.scatter(img[:,0], img[:,1], marker="x", s=90, label="outputs")
    for p, q in zip(pts, img):
        plt.plot([p[0], q[0]], [p[1], q[1]], linewidth=1)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.grid(True, alpha=0.3)
    plt.title(title)
    plt.legend()
    plt.show()

plot_before_after(P, points, "Projection: many points collapse to the x-axis")


### Student task

Add three more points with the same $x$-coordinate but different $y$-coordinates. Confirm that they all have the same output.


In [ ]:
# TODO: modify this array and run the cell
my_points = np.array([[2, -5],
                      [2, 0],
                      [2, 8]], dtype=float)

print(my_points @ P.T)


## 2. Column space: the world of reachable outputs

The column space of $A$ is all possible outputs $Ax$.

For

$$
A=\begin{bmatrix}1&2\\2&4\end{bmatrix},
$$

the two columns point in the same direction. The output world is only a line.


In [ ]:
A = np.array([[1, 2],
              [2, 4]], dtype=float)

print("A =")
print(A)
print("rank(A) =", np.linalg.matrix_rank(A))
print("columns:")
print(A[:,0], A[:,1])


In [ ]:
def plot_column_space_2d(A, coeff_lim=2.5, n=25):
    c1, c2 = A[:,0], A[:,1]
    ts = np.linspace(-coeff_lim, coeff_lim, n)
    outputs = []
    for a in ts:
        for b in ts:
            outputs.append(a*c1 + b*c2)
    outputs = np.array(outputs)
    plt.figure(figsize=(7,7))
    plt.axhline(0, linewidth=1)
    plt.axvline(0, linewidth=1)
    plt.scatter(outputs[:,0], outputs[:,1], s=10, alpha=0.45, label="reachable outputs")
    plt.arrow(0,0,c1[0],c1[1], head_width=0.12, length_includes_head=True, label="column 1")
    plt.arrow(0,0,c2[0],c2[1], head_width=0.12, length_includes_head=True, label="column 2")
    plt.gca().set_aspect('equal', adjustable='box')
    plt.grid(True, alpha=0.3)
    plt.title("Column space as reachable outputs")
    plt.legend()
    plt.show()

plot_column_space_2d(A)


### Reachable or not?

A target $b$ is reachable if it belongs to the column space.

For the matrix above, $b$ is reachable exactly when it lies on the line spanned by $(1,2)$.


In [ ]:
targets = [np.array([3, 6.]), np.array([3, 5.]), np.array([-2, -4.])]

for b in targets:
    x, residuals, rank, s = np.linalg.lstsq(A, b, rcond=None)
    print("b =", b)
    print("least-squares x =", x)
    print("Ax =", A @ x)
    print("residual norm =", np.linalg.norm(A @ x - b))
    print()


## 3. Null space: invisible input directions

The null space is the set of all inputs sent to zero:

$$
\operatorname{Null}(A)=\{x:Ax=0\}.
$$

For this matrix, one null direction is $(-2,1)$.


In [ ]:
v = np.array([-2, 1.])
print("A @ v =", A @ v)

x0 = np.array([1, 0.])
for t in [-3, -1, 0, 2, 5]:
    x = x0 + t*v
    print(f"t={t:>3}: x={x}, Ax={A@x}")


In [ ]:
# Visualize that a whole line of inputs has the same output.
ts = np.linspace(-3, 3, 80)
line = np.array([x0 + t*v for t in ts])
outputs = line @ A.T

plt.figure(figsize=(7,7))
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.plot(line[:,0], line[:,1], label="inputs x0 + t v")
plt.scatter(outputs[:,0], outputs[:,1], s=15, label="outputs Ax")
plt.gca().set_aspect('equal', adjustable='box')
plt.grid(True, alpha=0.3)
plt.title("Moving along a null direction does not change the output")
plt.legend()
plt.show()


## 4. Rank-nullity in computation

For an $m\times n$ matrix,

$$
\operatorname{rank}(A)+\operatorname{nullity}(A)=n.
$$

The input dimension is split into surviving directions and disappearing directions.


In [ ]:
def rank_nullity_report(A, name="A", tol=1e-10):
    A = np.array(A, dtype=float)
    U, s, Vt = np.linalg.svd(A)
    rank = np.sum(s > tol)
    nullity = A.shape[1] - rank
    print(f"{name}.shape = {A.shape}")
    print("singular values =", s)
    print("rank =", rank)
    print("nullity =", nullity)
    print("rank + nullity =", rank + nullity)
    print("input dimension =", A.shape[1])
    return rank, nullity

rank_nullity_report(A, "A")


In [ ]:
examples = {
    "zero 2x2": np.zeros((2,2)),
    "rank one 2x2": np.array([[1,2],[2,4]]),
    "full rank 2x2": np.array([[1,2],[3,4]]),
    "wide 2x5": np.random.default_rng(1).normal(size=(2,5)),
    "tall 5x2": np.random.default_rng(2).normal(size=(5,2)),
}

for name, M in examples.items():
    print("---")
    rank_nullity_report(M, name)


### Student task

Create your own $3\times 4$ matrix with rank $2$. Verify that its nullity is $2$.

Hint: make the last two columns combinations of the first two columns.


In [ ]:
# TODO: build your own example
M = np.array([[1, 0, 1, 2],
              [0, 1, 1, -1],
              [1, 1, 2, 1]], dtype=float)
rank_nullity_report(M, "M")


## 5. Three solution types for $Ax=b$

For a linear system $Ax=b$, there are three common outcomes:

1. no solution,
2. exactly one solution,
3. infinitely many solutions.

The column space decides whether a solution exists. The null space decides whether it is unique.


In [ ]:
def solve_report(A, b, name="system"):
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    x, residuals, rank, s = np.linalg.lstsq(A, b, rcond=None)
    residual = np.linalg.norm(A @ x - b)
    nullity = A.shape[1] - np.linalg.matrix_rank(A)
    print(f"{name}")
    print("A =")
    print(A)
    print("b =", b)
    print("least-squares x =", x)
    print("Ax =", A @ x)
    print("residual norm =", residual)
    print("rank =", np.linalg.matrix_rank(A), "nullity =", nullity)
    if residual < 1e-8 and nullity == 0:
        print("Interpretation: consistent with a unique solution.")
    elif residual < 1e-8 and nullity > 0:
        print("Interpretation: consistent with infinitely many solutions.")
    else:
        print("Interpretation: inconsistent; no exact solution.")

solve_report(np.array([[1,2],[3,4]]), np.array([1,0]), "unique solution")
print()
solve_report(np.array([[1,2],[2,4]]), np.array([3,6]), "infinitely many solutions")
print()
solve_report(np.array([[1,2],[2,4]]), np.array([3,5]), "no solution")


## 6. Near information loss: numerical danger

A matrix can be technically invertible but almost lose information.

Consider

$$
A_\epsilon=\begin{bmatrix}1&1\\1&1+\epsilon\end{bmatrix}.
$$

When $\epsilon$ is very small, the columns are almost the same. The matrix is invertible, but solving backwards becomes extremely sensitive.


In [ ]:
def near_singular_demo(eps_values):
    b = np.array([2, 2.0001])
    for eps in eps_values:
        Aeps = np.array([[1, 1], [1, 1+eps]], dtype=float)
        cond = np.linalg.cond(Aeps)
        x = np.linalg.solve(Aeps, b)
        print(f"eps={eps:>10g}  cond={cond:>12.3e}  solution={x}")

near_singular_demo([1e-1, 1e-2, 1e-4, 1e-6, 1e-8])


In [ ]:
eps_values = np.logspace(-1, -10, 100)
conds = []
for eps in eps_values:
    Aeps = np.array([[1, 1], [1, 1+eps]], dtype=float)
    conds.append(np.linalg.cond(Aeps))

plt.figure(figsize=(7,5))
plt.loglog(eps_values, conds)
plt.gca().invert_xaxis()
plt.xlabel("epsilon")
plt.ylabel("condition number")
plt.title("As columns become almost dependent, solving becomes unstable")
plt.grid(True, which="both", alpha=0.3)
plt.show()


## 7. Images and controlled information loss

Now we use a small synthetic image. We will compress it by keeping only a few singular directions.

This is a preview of later chapters on SVD and image compression.


In [ ]:
# Create a synthetic image with structure
n = 80
x = np.linspace(-3, 3, n)
y = np.linspace(-3, 3, n)
X, Y = np.meshgrid(x, y)
image = np.exp(-(X**2 + Y**2)) + 0.7*np.exp(-((X-1.2)**2 + (Y+0.8)**2)/0.4)
image += 0.15*np.sin(5*X) * np.cos(3*Y)

plt.figure(figsize=(5,5))
plt.imshow(image, cmap='gray')
plt.title("Synthetic image as a matrix")
plt.axis('off')
plt.show()


In [ ]:
U, s, Vt = np.linalg.svd(image, full_matrices=False)

def rank_k_approx(k):
    return (U[:, :k] * s[:k]) @ Vt[:k, :]

for k in [1, 3, 5, 10, 20, 40]:
    approx = rank_k_approx(k)
    err = np.linalg.norm(image - approx) / np.linalg.norm(image)
    plt.figure(figsize=(5,5))
    plt.imshow(approx, cmap='gray')
    plt.title(f"Rank {k} approximation, relative error {err:.3f}")
    plt.axis('off')
    plt.show()


### Reflection

Compression deliberately throws away directions. The goal is not to keep every piece of information, but to keep the most meaningful information.

Later, singular values will tell us which directions matter most.


## 8. High-dimensional random projections

A random matrix from $\mathbb{R}^{100}$ to $\mathbb{R}^{20}$ must lose information because the output dimension is smaller than the input dimension.

But it may still preserve enough information for a particular task.


In [ ]:
rng = np.random.default_rng(42)
A_hd = rng.normal(size=(20, 100)) / np.sqrt(20)
rank_nullity_report(A_hd, "A_hd")

# Create two clouds in 100D and project them to 20D
n_samples = 300
cloud1 = rng.normal(loc=0.0, scale=1.0, size=(n_samples, 100))
cloud2 = rng.normal(loc=0.25, scale=1.0, size=(n_samples, 100))

proj1 = cloud1 @ A_hd.T
proj2 = cloud2 @ A_hd.T

# Compare pairwise distances for matched pairs
original_dist = np.linalg.norm(cloud1 - cloud2, axis=1)
projected_dist = np.linalg.norm(proj1 - proj2, axis=1)

plt.figure(figsize=(6,5))
plt.scatter(original_dist, projected_dist, s=15, alpha=0.6)
plt.xlabel("distance in 100D")
plt.ylabel("distance after projection to 20D")
plt.title("Random projection loses dimensions but may preserve distance patterns")
plt.grid(True, alpha=0.3)
plt.show()

print("correlation between original and projected matched distances:",
      np.corrcoef(original_dist, projected_dist)[0,1])


## 9. Final mini-project

Choose one of the following.

### Option A: design a measurement system

Create a matrix $A$ that maps $\mathbb{R}^4$ to $\mathbb{R}^2$.

1. Compute its rank and nullity.
2. Interpret what information it keeps.
3. Interpret what information it loses.
4. Find two different inputs that produce the same output.

### Option B: compare exact loss and approximate loss

Create a $50\times 50$ image-like matrix.

1. Compute rank-$k$ approximations for several $k$.
2. Plot the relative error versus $k$.
3. Explain when the image still looks good and when too much information is lost.

### Option C: near-singular systems

Create a family of $2\times 2$ matrices whose columns become almost dependent.

1. Plot the condition number.
2. Solve $Ax=b$ for several values of the parameter.
3. Explain why the solution becomes unstable.
